# Orbital Mission Control — 3D Satellite Simulation

This project is an early prototype of a 3D orbital visualization system developed as part of a future NASA-inspired Mission Control platform.

The simulation uses Three.js to create an interactive 3D space environment containing a textured Earth, a star field, an inclined satellite orbit, and a moving satellite model.

The current system includes:

* A textured 3D Earth
* Dynamic lighting representing illumination from the Sun
* A surrounding star field
* Interactive camera controls using OrbitControls
* A satellite model with solar panels
* A circular inclined orbital plane
* A visible orbital trajectory
* A position vector from the Earth's center to the satellite
* Real-time satellite motion based on angular velocity
* Continuous Earth and satellite rotation
* Responsive rendering for different screen sizes

The satellite currently follows a simplified circular orbital model:

$$
x = r\cos(\theta)
$$

$$
y = r\sin(\theta)
$$

where \(r\) is the orbital radius and \(\theta\) is the satellite's orbital angle.

The orbital plane is then rotated by an inclination angle to represent a non-equatorial orbit.

This prototype does not yet use real satellite tracking data. It is designed as the visualization foundation for a larger system that will eventually integrate orbital mechanics, real satellite data, coordinate transformations, SGP4 propagation, telemetry, databases, and real-time data streams.

The long-term goal is to integrate this visualization into a web-based Mission Control platform capable of displaying real-time orbital and spacecraft-related information.

Current pipeline:

Satellite parameters
→ Orbital position calculation
→ 3D coordinate transformation
→ Three.js visualization
→ Real-time animation


In [15]:
%%html
<div id="earth-canvas-container"></div>
<script type="module">
import * as THREE from "https://unpkg.com/three@0.126.0/build/three.module.js";
import { OrbitControls } from "https://unpkg.com/three@0.126.0/examples/jsm/controls/OrbitControls.js";

const container = document.getElementById("earth-canvas-container");

// --------------------------------------------------
// 1. SCENE
// --------------------------------------------------
const scene = new THREE.Scene();
scene.background = new THREE.Color(0x000005);

// --------------------------------------------------
// 2. CAMERA
// --------------------------------------------------
const camera = new THREE.PerspectiveCamera(
    45,
    600 / 400,
    0.1,
    1000
);
camera.position.set(0, 0, 3);

// --------------------------------------------------
// 3. RENDERER
// --------------------------------------------------
const renderer = new THREE.WebGLRenderer({
    antialias: true
});
renderer.setSize(600, 400);
renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
container.appendChild(renderer.domElement);

// --------------------------------------------------
// 4. EARTH GEOMETRY
// --------------------------------------------------
const earthGeometry = new THREE.SphereGeometry(1, 128, 128);

// --------------------------------------------------
// 5. EARTH TEXTURE
// --------------------------------------------------
const textureLoader = new THREE.TextureLoader();
const earthTexture = textureLoader.load(
    "https://threejs.org/examples/textures/planets/earth_atmos_2048.jpg"
);

// --------------------------------------------------
// 6. EARTH MATERIAL
// --------------------------------------------------
const earthMaterial = new THREE.MeshPhongMaterial({
    map: earthTexture
});

// --------------------------------------------------
// 7. EARTH MESH
// --------------------------------------------------
const earth = new THREE.Mesh(earthGeometry, earthMaterial);
scene.add(earth);

// --------------------------------------------------
// 8. LIGHTING
// --------------------------------------------------
const ambientLight = new THREE.AmbientLight(0xffffff, 0.25);
scene.add(ambientLight);

const sunLight = new THREE.DirectionalLight(0xffffff, 2);
sunLight.position.set(5, 3, 5);
scene.add(sunLight);

// --------------------------------------------------
// 9. STARS
// --------------------------------------------------
const starGeometry = new THREE.BufferGeometry();
const starCount = 5000;
const starPositions = new Float32Array(starCount * 3);

for (let i = 0; i < starCount * 3; i++) {
    starPositions[i] = (Math.random() - 0.5) * 200;
}

starGeometry.setAttribute(
    "position",
    new THREE.BufferAttribute(starPositions, 3)
);

const starMaterial = new THREE.PointsMaterial({
    color: 0xffffff,
    size: 0.05
});

const stars = new THREE.Points(starGeometry, starMaterial);
scene.add(stars);

// --------------------------------------------------
// 10. CAMERA CONTROLS
// --------------------------------------------------
const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;
controls.minDistance = 1.5;
controls.maxDistance = 10;

// --------------------------------------------------
// 11. ANIMATION LOOP
// --------------------------------------------------
function animate() {
    requestAnimationFrame(animate);
    earth.rotation.y += 0.0015;
    stars.rotation.y += 0.00005;
    controls.update();
    renderer.render(scene, camera);
}

animate();
</script>

In [18]:
%%html
<div id="earth-container-new"></div>
<script type="module">
import * as THREE from "https://unpkg.com/three@0.126.0/build/three.module.js";
import { OrbitControls } from "https://unpkg.com/three@0.126.0/examples/jsm/controls/OrbitControls.js";

const container = document.getElementById("earth-container-new");

// --------------------------------------------------
// 1. SCENE
// --------------------------------------------------
const scene = new THREE.Scene();
scene.background = new THREE.Color(0x000005);

// --------------------------------------------------
// 2. CAMERA
// --------------------------------------------------
const camera = new THREE.PerspectiveCamera(
    45,
    600 / 400,
    0.1,
    1000
);
camera.position.set(0, 2, 4);

// --------------------------------------------------
// 3. RENDERER
// --------------------------------------------------
const renderer = new THREE.WebGLRenderer({
    antialias: true
});
renderer.setSize(600, 400);
renderer.setPixelRatio(Math.min(window.devicePixelRatio, 2));
container.appendChild(renderer.domElement);

// --------------------------------------------------
// 4. EARTH GEOMETRY & TEXTURE
// --------------------------------------------------
const earthGeometry = new THREE.SphereGeometry(1, 128, 128);
const textureLoader = new THREE.TextureLoader();
const earthTexture = textureLoader.load(
    "https://threejs.org/examples/textures/planets/earth_atmos_2048.jpg"
);
const earthMaterial = new THREE.MeshPhongMaterial({
    map: earthTexture
});
const earth = new THREE.Mesh(earthGeometry, earthMaterial);
scene.add(earth);

// --------------------------------------------------
// 5. HELPER AXES (Yörünge ve Eksen Yardımcıları)
// --------------------------------------------------
// Kırmızı: X, Yeşil: Y, Mavi: Z eksenini gösterir
const axesHelper = new THREE.AxesHelper(1.8);
scene.add(axesHelper);

// --------------------------------------------------
// 6. SATELLITE & GLOW MARKER (Uydu ve Parlak İşaretçi)
// --------------------------------------------------
// Ana uydu gövdesi (küçük metalik küre)
const satelliteGeometry = new THREE.SphereGeometry(0.04, 16, 16);
const satelliteMaterial = new THREE.MeshStandardMaterial({
    color: 0xcccccc,
    metalness: 0.8,
    roughness: 0.2
});
const satellite = new THREE.Mesh(satelliteGeometry, satelliteMaterial);

// Parlak İşaretçi (Uyduyu daha görünür kılan yarı saydam kırmızı aura)
const glowGeometry = new THREE.SphereGeometry(0.09, 16, 16);
const glowMaterial = new THREE.MeshBasicMaterial({
    color: 0xff3333,
    transparent: true,
    opacity: 0.5,
    blending: THREE.AdditiveBlending
});
const glowMarker = new THREE.Mesh(glowGeometry, glowMaterial);
satellite.add(glowMarker); // İşaretçiyi uydunun içine ekliyoruz

scene.add(satellite);

// --------------------------------------------------
// 7. SATELLITE ORBIT PATH (Yörünge Çizgisi)
// --------------------------------------------------
const orbitRadius = 1.6;
const orbitSegments = 128;
const orbitGeometry = new THREE.BufferGeometry();
const orbitPositions = [];

for (let i = 0; i <= orbitSegments; i++) {
    const theta = (i / orbitSegments) * Math.PI * 2;
    // Yörüngeyi hafif eğik (ecliptic/inclined) yapmak için x ve z eksenine yerleştiriyoruz
    orbitPositions.push(Math.cos(theta) * orbitRadius, 0.2 * Math.sin(theta), Math.sin(theta) * orbitRadius);
}

orbitGeometry.setAttribute(
    "position",
    new THREE.Float32BufferAttribute(orbitPositions, 3)
);

const orbitMaterial = new THREE.LineBasicMaterial({
    color: 0x00ffcc,
    transparent: true,
    opacity: 0.4
});
const orbitLine = new THREE.Line(orbitGeometry, orbitMaterial);
scene.add(orbitLine);

// --------------------------------------------------
// 8. LIGHTING
// --------------------------------------------------
const ambientLight = new THREE.AmbientLight(0xffffff, 0.25);
scene.add(ambientLight);

const sunLight = new THREE.DirectionalLight(0xffffff, 2);
sunLight.position.set(5, 3, 5);
scene.add(sunLight);

// --------------------------------------------------
// 9. STARS BACKGROUND
// --------------------------------------------------
const starGeometry = new THREE.BufferGeometry();
const starCount = 3000;
const starPositions = new Float32Array(starCount * 3);
for (let i = 0; i < starCount * 3; i++) {
    starPositions[i] = (Math.random() - 0.5) * 200;
}
starGeometry.setAttribute(
    "position",
    new THREE.BufferAttribute(starPositions, 3)
);
const starMaterial = new THREE.PointsMaterial({
    color: 0xffffff,
    size: 0.05
});
const stars = new THREE.Points(starGeometry, starMaterial);
scene.add(stars);

// --------------------------------------------------
// 10. CAMERA CONTROLS
// --------------------------------------------------
const controls = new OrbitControls(camera, renderer.domElement);
controls.enableDamping = true;
controls.minDistance = 1.3;
controls.maxDistance = 8;

// --------------------------------------------------
// 11. ANIMATION & ROTATION SPEED
// --------------------------------------------------
// Gerçek Dünya açısal hızı: ~7.292e-5 rad/s.
// Görselleştirme amacıyla bu fiziksel değeri fark edebilmemiz için 200 kat ölçeklendiriyoruz.
const realAngularVelocity = 7.292e-5;
const speedScale = 200;
const earthRotationSpeed = realAngularVelocity * speedScale;

let orbitAngle = 0;
const clock = new THREE.Clock();

function animate() {
    requestAnimationFrame(animate);
    const delta = clock.getDelta();

    // Dünya'nın kendi ekseni etrafında gerçekçi hıza bağlı dönmesi
    earth.rotation.y += earthRotationSpeed * delta * 60;

    // Uydunun yörüngedeki hareketi
    orbitAngle += 0.015;
    satellite.position.x = Math.cos(orbitAngle) * orbitRadius;
    satellite.position.y = 0.2 * Math.sin(orbitAngle);
    satellite.position.z = Math.sin(orbitAngle) * orbitRadius;

    // Yıldızların çok hafif arka plan hareketi
    stars.rotation.y += 0.00002;

    controls.update();
    renderer.render(scene, camera);
}

animate();
</script>